# Detect Source Cluster Egress IP

This notebook detects the public IP address used by this cluster for outbound connections.

**Purpose:** Identify the IP that needs to be whitelisted on target workspace

**Usage:**
1. Run this notebook on source workspace
2. IP is automatically saved to UC volume
3. Cleanup script will auto-detect from saved metadata

---

## Configuration

Update the volume path below to match your environment.

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THIS PATH
# ============================================================================

# Your UC volume path for storing migration artifacts
# Format: /Volumes/<catalog>/<schema>/<volume>
VOLUME_PATH = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/dashboard_migration"

# ============================================================================

print(f"Volume Path: {VOLUME_PATH}")
if "YOUR_CATALOG" in VOLUME_PATH:
    print("\nWARNING: Please update VOLUME_PATH with your actual UC volume path!")

In [ ]:
# Cell 1: Detect Cluster Egress IP

import requests
import json

print("="*80)
print("DETECTING CLUSTER EGRESS IP")
print("="*80)
print()

cluster_ip = None

# Method 1: Use ipify.org (most reliable)
try:
    print("Method 1: Using ipify.org...")
    response = requests.get('https://api.ipify.org?format=json', timeout=10)
    cluster_ip = response.json()['ip']
    print(f"   Detected IP: {cluster_ip}")
    print()
except Exception as e:
    print(f"   Failed: {e}")
    cluster_ip = None

# Method 2: Use icanhazip.com (backup)
if not cluster_ip:
    try:
        print("Method 2: Using icanhazip.com...")
        response = requests.get('https://icanhazip.com', timeout=10)
        cluster_ip = response.text.strip()
        print(f"   Detected IP: {cluster_ip}")
        print()
    except Exception as e:
        print(f"   Failed: {e}")
        cluster_ip = None

# Method 3: Use ipinfo.io (backup)
if not cluster_ip:
    try:
        print("Method 3: Using ipinfo.io...")
        response = requests.get('https://ipinfo.io/json', timeout=10)
        cluster_ip = response.json()['ip']
        print(f"   Detected IP: {cluster_ip}")
        print()
    except Exception as e:
        print(f"   Failed: {e}")
        cluster_ip = None

if cluster_ip:
    print("="*80)
    print("CLUSTER IP DETECTED")
    print("="*80)
    print()
    print(f"Cluster Egress IP: {cluster_ip}")
    print()
    
    # Calculate IP ranges
    parts = cluster_ip.split('.')
    print("IP Range Options for Whitelisting:")
    print(f"   Single IP (/32):   {cluster_ip}/32          (Most restrictive - recommended)")
    print(f"   Small range (/28): {parts[0]}.{parts[1]}.{parts[2]}.{int(parts[3]) // 16 * 16}/28   (16 IPs)")
    print(f"   Large range (/24): {parts[0]}.{parts[1]}.{parts[2]}.0/24    (256 IPs)")
    print()
    
    # Store in dbutils widget for programmatic access
    try:
        dbutils.widgets.text("cluster_ip", cluster_ip, "Detected Cluster IP")
        print("IP stored in widget 'cluster_ip'")
        print()
    except:
        pass
    
else:
    print("="*80)
    print("COULD NOT DETECT IP")
    print("="*80)
    print()
    print("Possible causes:")
    print("  - No internet access from cluster")
    print("  - Firewall blocking outbound connections")
    print("  - Network connectivity issues")
    print()
    raise Exception("Failed to detect cluster IP - cannot continue")

In [ ]:
# Cell 2: Save IP to Unity Catalog Volume (Optional but recommended)

if cluster_ip:
    print("="*80)
    print("SAVING IP METADATA TO UC VOLUME")
    print("="*80)
    print()
    
    import json
    from datetime import datetime
    
    # Configuration
    ip_file = f"{VOLUME_PATH}/cluster_ip_metadata.json"
    
    # Create metadata
    metadata = {
        "cluster_ip": cluster_ip,
        "detected_at": datetime.now().isoformat(),
        "cluster_id": spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "unknown"),
        "workspace_url": spark.conf.get("spark.databricks.workspaceUrl", "unknown"),
        "detected_by": spark.conf.get("spark.databricks.clusterUsageTags.clusterOwnerEmail", "unknown"),
        "suggested_ranges": {
            "single_ip": f"{cluster_ip}/32",
            "small_range": f"{'.'.join(cluster_ip.split('.')[:3])}.{int(cluster_ip.split('.')[3]) // 16 * 16}/28",
            "large_range": f"{'.'.join(cluster_ip.split('.')[:3])}.0/24"
        }
    }
    
    print(f"Target location: {ip_file}")
    print()
    
    save_success = False
    save_method = None
    errors = []
    
    # Method 1: Use dbutils.fs.put (preferred for UC volumes)
    try:
        print("Method 1: Saving via dbutils.fs.put...")
        metadata_json = json.dumps(metadata, indent=2)
        dbutils.fs.put(ip_file, metadata_json, overwrite=True)
        save_success = True
        save_method = "dbutils.fs.put"
        print("   SUCCESS")
        print()
    except Exception as e1:
        errors.append(f"Method 1: {e1}")
        print(f"   Failed: {e1}")
        print()
        
        # Method 2: Use /dbfs mount point
        try:
            print("Method 2: Saving via /dbfs mount...")
            dbfs_path = ip_file.replace("/Volumes", "/dbfs/Volumes")
            
            # Ensure directory exists
            import os
            os.makedirs(os.path.dirname(dbfs_path), exist_ok=True)
            
            with open(dbfs_path, 'w') as f:
                json.dump(metadata, f, indent=2)
            save_success = True
            save_method = "/dbfs mount"
            print("   SUCCESS")
            print()
        except Exception as e2:
            errors.append(f"Method 2: {e2}")
            print(f"   Failed: {e2}")
            print()
    
    # Report results
    if save_success:
        print("="*80)
        print(f"METADATA SAVED SUCCESSFULLY")
        print("="*80)
        print()
        print(f"Method used: {save_method}")
        print(f"File location: {ip_file}")
        print()
        print("Metadata contents:")
        print(json.dumps(metadata, indent=2))
        print()
        
        # Verify read access
        try:
            verify_content = dbutils.fs.head(ip_file)
            if cluster_ip in verify_content:
                print("Verification: File readable and contains correct IP")
                print()
        except Exception as e:
            print(f"Verification warning: {e}")
            print("   File may have been saved but is not readable")
            print()
        
        print("The cleanup script can now auto-detect this IP")
        print()
    else:
        print("="*80)
        print("WARNING: COULD NOT SAVE IP METADATA")
        print("="*80)
        print()
        print("All save methods failed:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        print()
        print(f"Detected IP: {cluster_ip}")
        print(f"Target: {ip_file}")
        print()
        print("Possible causes:")
        print("  - No WRITE permission on UC volume")
        print("  - Volume path does not exist")
        print("  - Volume not mounted on cluster")
        print()
        print("To fix:")
        print("  1. Check volume permissions:")
        print(f"     SHOW GRANTS ON VOLUME <your_catalog>.<your_schema>.<your_volume>;")
        print("  2. Grant write access if needed:")
        print(f"     GRANT ALL PRIVILEGES ON VOLUME ... TO `your-user-or-sp`;")
        print()
        print("You can still use the detected IP manually:")
        print(f"   ./scripts/auto_setup_ip_acl.sh --cluster-ip {cluster_ip}")
        print(f"   ./scripts/cleanup_ip_acl.sh --cluster-ip {cluster_ip}")
        print()

## Next Steps

After detecting the IP, add it to your target workspace:

### Option A: Using the automated script (from local terminal)

```bash
./scripts/auto_setup_ip_acl.sh --cluster-ip YOUR_IP_HERE --target-profile target-workspace
```

### Option B: Using CLI manually (from local terminal)

```bash
databricks ip-access-lists create \
  --label "source-workspace-migration" \
  --list-type ALLOW \
  --ip-addresses "YOUR_IP_HERE/32" \
  --profile target-workspace
```

### Option C: Via target workspace UI

1. Open target workspace
2. Go to Settings → Security → IP Access Lists
3. Add new entry with the detected IP